# AI Coding Agents in Machine Learning: A Specification-Driven Approach

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konstantin-schekotihin/dl_course/blob/master/AI-ML/08_ml_agents.ipynb)


In [ ]:
# Setup & configuration (skip in presentation slides)
%matplotlib inline
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import wandb

# Ensure headless W&B logging for automated tests and non-interactive sessions
os.environ["WANDB_MODE"] = os.environ.get("WANDB_MODE", "disabled")
os.environ["WANDB_SILENT"] = "true"

# Pin random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load course helper utilities
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/AI-ML/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp


## 1. Leverage and Defensibility in Agentic AI

- Modern AI coding assistants (**Antigravity / Agy, GitHub Copilot, Claude Code**) accelerate exploratory data analysis and model implementation.
- **The Course GenAI Policy & Defensio Requirement** (see `modalities.md`):
  - **Transparency**: You are permitted to use AI coding tools, provided you explicitly disclose and attribute them in your final report.
  - **Defensibility (60% Oral Exam)**: An AI assistant cannot sit your exam. You must personally defend and explain every mathematical principle, line of code, and hyperparameter choice in your project.
  - **Analytical Mastery**: The machine assists with typing code; **you** are the machine learning engineer responsible for problem formulation, validation, and conclusions!


### The Specification-Driven Engineering Paradigm

- **The Unconstrained Prompting Trap**:
  - Vague prompts (e.g. *"Build an image classifier for my dataset"*) trigger severe failure modes: silent data leakage, metric mismatches, and monolithic unmaintainable code.
- **The Engineering Solution**:
  - Ground all agent workflows in Tom Mitchell's foundational **Learning Task $(T, P, E)$** specification:
    1. **Constitution & Invariants**: Establish non-negotiable data and software engineering rules.
    2. **Task Specification & Clarification**: Define $(T, P, E)$ and resolve edge cases.
    3. **Architectural Planning**: Decouple data, models, training engines, and tracking.
    4. **Phased Tasks (TDD)**: Write test assertions before implementing neural modules.
    5. **MLOps Tracking**: Centralize metrics, curves, and confusion matrices in Weights & Biases.


## 2. Stage 1: The Constitution & Non-Negotiable Invariants

- Before requesting code from an agent, establish the **Constitution**—a set of immutable axioms governing engineering and statistical integrity:
  - **Machine Learning & Data Invariants**:
    1. **Strict Anti-Leakage**: Normalization statistics (means $\boldsymbol{\mu}$, stds $\boldsymbol{\sigma}$) must strictly be computed on the training split only.
    2. **Augmentation Isolation**: Stochastic augmentations must strictly occur on-the-fly during training batches, never offline before splitting.
    3. **Baseline-First**: Evaluate Dummy (random guess) and Linear baselines before building deep networks.
    4. **Determinism**: Pinned random seeds across PyTorch, NumPy, and DataLoaders.
  - **Software Engineering & MLOps Invariants**:
    1. **Isolated Virtual Environment**: Execute exclusively within `.venv` via `uv run python`.
    2. **Centralized MLOps Tracking**: Log all hyperparameters, curves, and artifacts to Weights & Biases (W&B).
    3. **Component Decoupling**: Decouple data loading, architectures, training loops, and evaluation.


### The Anti-Leakage Normalization Invariant

- Image pixel normalization shifts and scales channels:
  $$\boldsymbol{\mu}_{\text{train}} = \frac{1}{|\mathcal{D}_{\text{train}}|} \sum_{\mathbf{x} \in \mathcal{D}_{\text{train}}} \mathbf{x}, \quad \boldsymbol{\sigma}_{\text{train}} = \sqrt{\frac{1}{|\mathcal{D}_{\text{train}}|} \sum_{\mathbf{x} \in \mathcal{D}_{\text{train}}} (\mathbf{x} - \boldsymbol{\mu}_{\text{train}})^2}$$
  $$\mathbf{x}_{\text{val}}^{\text{norm}} = \frac{\mathbf{x}_{\text{val}} - \boldsymbol{\mu}_{\text{train}}}{\boldsymbol{\sigma}_{\text{train}} + \epsilon}, \quad \mathbf{x}_{\text{test}}^{\text{norm}} = \frac{\mathbf{x}_{\text{test}} - \boldsymbol{\mu}_{\text{train}}}{\boldsymbol{\sigma}_{\text{train}} + \epsilon}$$

> ⚠️ **Important:** Computing normalization statistics across the entire dataset prior to splitting leaks test distribution information into the model, producing invalid, overly optimistic evaluations!


In [ ]:
# Retrieve CIFAR-10 automatically via PyTorch, following the MNIST pattern
import torchvision.datasets as datasets

_cifar_path = (
    "../shared/data" if os.path.exists("../shared/data") else
    ("shared/data" if os.path.exists("shared/data") else "./data")
)

raw_train = datasets.CIFAR10(_cifar_path, train=True, download=True, transform=T.ToTensor())
raw_test = datasets.CIFAR10(_cifar_path, train=False, download=True, transform=T.ToTensor())

# Subsample for swift interactive in-slide execution
train_idx = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(42))[:1000]
test_idx = torch.randperm(len(raw_test), generator=torch.Generator().manual_seed(42))[:200]

cifar_imgs = torch.stack([raw_train[i][0] for i in train_idx])
cifar_labels = torch.tensor([raw_train[i][1] for i in train_idx])
test_imgs = torch.stack([raw_test[i][0] for i in test_idx])
test_labels = torch.tensor([raw_test[i][1] for i in test_idx])

# 1. Strict partition: 800 train, 200 val
train_imgs, val_imgs = cifar_imgs[:800], cifar_imgs[800:]
train_labels, val_labels = cifar_labels[:800], cifar_labels[800:]

# 2. Compute channel statistics strictly on the training partition stack
channel_mean = train_imgs.mean(dim=(0, 2, 3))
channel_std = train_imgs.std(dim=(0, 2, 3))


In [ ]:
# Concrete numerical verification: inspect computed training statistics
print(f"Training split shape:   {train_imgs.shape}")
print(f"Validation split shape: {val_imgs.shape}")
print(f"Computed Channel Mean:  {channel_mean.tolist()}")
print(f"Computed Channel Std:   {channel_std.tolist()}")

# Apply training statistics to normalize train, val, and test splits
norm = T.Normalize(mean=channel_mean, std=channel_std)
X_train = norm(train_imgs)
X_val = norm(val_imgs)
X_test = norm(test_imgs)

print("Verification complete: test data remained completely unobserved during normalization.")


## 3. Stage 2: Specifying the Learning Task & Analytical Clarification

- In specification-driven workflows, define the learning task via Mitchell's triad $(T, P, E)$:
  - **Task ($T$)**:
    - Ingest an RGB image tensor $\mathbf{x} \in \mathbb{R}^{3 \times 32 \times 32}$.
    - Output class prediction $\hat{t} \in \{0, \dots, 9\}$ across 10 object classes:  
      *{airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck}*.
  - **Performance ($P$)**:
    - Primary Metric: Top-1 Test Accuracy $\ge 75.0\%$.
    - Secondary Metric: Macro-averaged $F_1$-score (verifying balanced performance across all 10 classes).
    - Baselines: Dummy Baseline ($10.0\%$), Linear Softmax Regression ($\ge 38.0\%$).
    - Operational: CPU inference latency $\le 15\text{ms}$ per image.
  - **Experience ($E$)**:
    - 60,000 $32 \times 32$ color images partitioned into strict disjoint subsets: 40k train, 10k val, 10k test.

<small>📖 *Source: [Mitchell, T. M. (1997). Machine Learning. McGraw-Hill](https://www.cs.cmu.edu/~tom/)*</small>


### Analytical Clarification: Resolving Ambiguities

- Proactively clarify hidden design decisions before generating code:
  1. **Loss Function Formulation**:
     - Multi-class cross-entropy error function:
       $$E(\mathbf{w}) = -\sum_{k=1}^K t_k \ln y_k$$
     - Implemented via `nn.CrossEntropyLoss()` (combining `LogSoftmax` and `NLLLoss` for numerical stability).
  2. **Validation Discipline & Early Stopping**:
     - Evaluate metrics at the end of every epoch; save the checkpoint with the lowest validation loss to avoid overfitting.
  3. **Class Balance Verification**:
     - Verify equal class distribution (6,000 per class); report confusion matrices to inspect pair-wise semantic confusion (e.g. *cat* vs. *dog*).

> 💡 **Key Takeaway:** Resolving ambiguities upfront prevents the agent from making arbitrary, incorrect assumptions during code generation!


## 4. Stage 3 & 4: Architectural Planning & Phased Tasks

- **Component Decoupling Contract**:
  - `data_pipeline.py`: Dataset loading, partitioning, leak-free normalization transforms.
  - `models/baselines.py`: `DummyClassifier` and `LinearSoftmaxRegression`.
  - `models/convnet.py`: Modular convolutional building blocks (`VGGBlock`) and composed `ModularVGGNet`.
  - `engine.py`: Stateless training epoch and evaluation functions.
  - `tracking.py`: Weights & Biases (W&B) logging wrappers for metrics, loss curves, and confusion matrix artifacts.
  - `helpers.py`: Graphical canvas and layout styling.

- **Ordered Phased Tasks (Blocking Prerequisites)**:
  - **Phase 1**: Data integrity & leak-free transforms (*Blocking prerequisite*).
  - **Phase 2**: Baseline models & reference benchmarks (*Blocking prerequisite*).
  - **Phase 3**: Modular VGG architecture & TDD unit tests.
  - **Phase 4**: W&B experiment tracking & hyperparameter tuning.
  - **Phase 5**: Final evaluation & error analysis report.


### Pre-Execution Logical Audit

- Perform a static code audit before executing agent-generated modules:

| Failure Mode | What the Agent Generated | Statistical / SE Consequence | Audit Remediation |
| :--- | :--- | :--- | :--- |
| **Silent Normalization Leakage** | Computed `mean = dataset.data.mean()` across entire dataset before split. | Validation and test distributions leak into training. | Enforce calculation strictly on the training subset tensor. |
| **Augmentation Before Splitting** | Applied `RandomHorizontalFlip` to raw dataset before `random_split`. | Augmented twins of identical images appear in both train and test. | Apply data augmentation on-the-fly strictly during training batches. |
| **Evaluation Mode Omission** | Ran validation loop without `model.eval()` and `torch.no_grad()`. | Dropout remains active; gradient graph wastes GPU/CPU memory. | Enforce `model.eval()` and `with torch.no_grad():` in evaluation. |
| **Missing Baselines** | Built deep CNN immediately and reported 72% accuracy in isolation. | Lacks context: does convolution provide lift over linear models? | Implement Dummy (10%) and Linear (39%) benchmarks first. |


## 5. Baseline-First: Establishing Reference Benchmarks

- **The Baseline-First Invariant**:
  - Never evaluate a complex neural architecture in isolation. Establish benchmarks first to quantify architectural lift.
- **Reference Benchmarks**:
  1. **Dummy Uniform Classifier**:
     - Chooses uniformly at random among $K=10$ classes.
     - Expected Top-1 Accuracy: $10.0\%$.
  2. **Linear Softmax Regression**:
     - Single linear layer mapping flattened pixels $\mathbf{x} \in \mathbb{R}^{3072}$ to 10 class logits:
       $$\mathbf{a} = \mathbf{W}\mathbf{x} + \mathbf{b}, \quad y_k = \frac{\exp(a_k)}{\sum_j \exp(a_j)}$$
     - Benchmark Top-1 Accuracy on CIFAR-10: $\approx 38\text{--}40\%$.

<small>📖 *Source: [Bishop, C. M., & Bishop, H. (2024). Deep Learning: Foundations and Concepts. Springer](https://www.bishopbook.com/)*</small>


In [ ]:
# 1. Baseline 1: Dummy Uniform Classifier
dummy_preds = torch.randint(0, 10, (len(test_labels),))
dummy_acc = (dummy_preds == test_labels).float().mean().item()

# 2. Baseline 2: Linear Softmax Regression
class LinearSoftmaxRegression(nn.Module):
    def __init__(self, in_features=3 * 32 * 32, num_classes=10):
        super().__init__()
        self.linear = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.linear(x.flatten(start_dim=1))

linear_model = LinearSoftmaxRegression()
optimizer_lin = optim.SGD(linear_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    optimizer_lin.zero_grad()
    out = linear_model(X_train)
    loss = criterion(out, train_labels)
    loss.backward()
    optimizer_lin.step()

with torch.no_grad():
    linear_val_preds = linear_model(X_val).argmax(dim=1)
    linear_acc = (linear_val_preds == val_labels).float().mean().item()

print(f"Dummy Classifier Accuracy:        {dummy_acc * 100:.2f}% (random guess ~10%)")
print(f"Linear Softmax Baseline Accuracy: {linear_acc * 100:.2f}% (trained benchmark)")


## 6. Stage 6: Test-Driven Development (TDD) for Deep Networks

- In professional practice, Test-Driven Development applies directly to deep learning pipelines.
- **Composition over Monolithic Duplication**:
  - Instead of writing a rigid 50-line network definition, implement a clean, reusable **VGGBlock**:
    $$\mathbf{h}^{(l)} = \mathrm{MaxPool}(\mathrm{ReLU}(\mathbf{W} * \mathbf{h}^{(l-1)} + \mathbf{b}))$$
  - Builds directly on convolutional feature maps and non-linear activations introduced in Lecture 04.
  - Stacking 3 modular blocks progressively halves spatial resolution while doubling channel capacity:
    $$32 \times 32 \times 3 \xrightarrow{\text{Block 1}} 16 \times 16 \times 32 \xrightarrow{\text{Block 2}} 8 \times 8 \times 64 \xrightarrow{\text{Block 3}} 4 \times 4 \times 128$$

<small>📖 *Source: [Simonyan, K., & Zisserman, A. (2015). Very Deep Convolutional Networks for Large-Scale Image Recognition. ICLR](https://arxiv.org/abs/1409.1556)*</small>


In [ ]:
def test_vgg_block_spatial_halving(block_cls):
    """TDD Precondition: VGGBlock must double channels and halve spatial dimensions."""
    block = block_cls(in_channels=3, out_channels=32)
    sample_input = torch.randn(4, 3, 32, 32)
    output = block(sample_input)
    
    assert output.shape == (4, 32, 16, 16), f"Expected (4, 32, 16, 16), got {output.shape}"
    assert not torch.isnan(output).any(), "Block produced NaN values"
    print(f"✓ VGGBlock assertion passed! Input {sample_input.shape} -> Output {output.shape}")

print("TDD test harness defined.")


In [ ]:
class VGGBlock(nn.Module):
    """
    Modular VGG-style block: Conv2d -> ReLU -> MaxPool2d.
    Extracts spatial feature maps and halves spatial dimensions.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


In [ ]:
# Verify VGGBlock against our TDD test assertion
test_vgg_block_spatial_halving(VGGBlock)

# Explicit inspection of feature activation statistics
sample_x = torch.randn(2, 3, 32, 32)
vgg_blk = VGGBlock(3, 32)
sample_out = vgg_blk(sample_x)
print(f"Sample output mean: {sample_out.mean().item():.4f}, std: {sample_out.std().item():.4f}")


In [ ]:
class ModularVGGNet(nn.Module):
    """
    Modular Convolutional Network composed of stacked VGGBlocks and an MLP head.
    Avoids monolithic code duplication and requires no complex normalization layers.
    """
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            VGGBlock(in_channels=3, out_channels=32),    # 32x32 -> 16x16
            VGGBlock(in_channels=32, out_channels=64),   # 16x16 -> 8x8
            VGGBlock(in_channels=64, out_channels=128),  # 8x8   -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


In [ ]:
# Instantiate model and test forward pass across diverse batch sizes
model = ModularVGGNet(num_classes=10)
model.eval()

for batch_size in [1, 4, 16]:
    dummy_batch = torch.randn(batch_size, 3, 32, 32)
    with torch.no_grad():
        logits = model(dummy_batch)
    print(f"Batch size {batch_size:2d} -> Logits tensor shape: {logits.shape}")

assert logits.shape == (16, 10), "Logit output shape mismatch!"


In [ ]:
# Sanity Test: A properly formulated model MUST memorize 16 samples to zero loss
torch.manual_seed(42)
test_model = ModularVGGNet(num_classes=10)
optimizer = optim.Adam(test_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

x_toy = torch.randn(16, 3, 32, 32)
y_toy = torch.randint(0, 10, (16,))
initial_loss = criterion(test_model(x_toy), y_toy).item()

for step in range(50):
    optimizer.zero_grad()
    loss = criterion(test_model(x_toy), y_toy)
    loss.backward()
    optimizer.step()

final_loss = loss.item()
print(f"Initial loss: {initial_loss:.4f} -> Final loss after 50 steps: {final_loss:.4f}")
assert final_loss < 0.10, "Model failed the single-batch memorization test!"
print("✓ Overfitting sanity check passed: network capacity and gradients verified.")


## 7. Stage 7: Dynamic Revisions & Generalization Gap

- **The Generalization Gap in Deep Learning**:
  - Training accuracy reaches $98.5\%$, while validation accuracy plateaus at $64.2\%$.
  - The model has memorized high-frequency training noise rather than invariant object semantics.
- **The Formal Revision Workflow**:
  - Rather than hacking code ad-hoc, initiate a formal **Plan Revision**:
    1. **Stochastic Augmentations**: Introduce `RandomHorizontalFlip` on-the-fly during training.
    2. **$L_2$ Regularization (Weight Decay)**: Penalize large weights via AdamW ($\lambda = 10^{-4}$):
       $$\tilde{E}(\mathbf{w}) = E(\mathbf{w}) + \frac{\lambda}{2} \|\mathbf{w}\|^2$$
    3. **Dropout Regularization**: Zero activations with probability $p=0.3$ to prevent co-adaptation.
  - The revised pipeline boosts validation accuracy to **$\ge 75\%$**, successfully closing the gap!


In [ ]:
# Training step with AdamW and L2 weight decay (Regularization Plan Revision)
torch.manual_seed(42)
vgg_model = ModularVGGNet(num_classes=10)
optimizer_reg = optim.AdamW(vgg_model.parameters(), lr=1e-3, weight_decay=1e-4)

train_dataset = TensorDataset(X_train, train_labels)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

vgg_model.train()
for epoch in range(5):
    total_loss = 0.0
    for bx, by in train_loader:
        optimizer_reg.zero_grad()
        logits = vgg_model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer_reg.step()
        total_loss += loss.item() * len(by)
    avg_loss = total_loss / len(train_dataset)

vgg_model.eval()
with torch.no_grad():
    val_logits = vgg_model(X_val)
    val_preds = val_logits.argmax(dim=1)
    vgg_val_acc = (val_preds == val_labels).float().mean().item()

print(f"Epoch 5 Average Train Loss: {avg_loss:.4f}")
print(f"ModularVGGNet Validation Accuracy: {vgg_val_acc * 100:.2f}%")


## 8. Stage 8: Centralized MLOps Tracking with Weights & Biases

- **Why ad-hoc terminal printing fails**:
  - Over a semester project, you will evaluate dozens of model configurations.
  - Centralized tracking in **Weights & Biases (W&B)** captures:
    1. **Hyperparameters**: learning rate, weight decay, batch size, random seed.
    2. **Epoch Metrics**: train/val loss curves, Top-1 accuracy, learning rate schedules.
    3. **Diagnostic Artifacts**: class-by-class confusion matrix heatmaps and classification reports.

> 💡 **Key Takeaway:** Centralized tracking guarantees reproducibility and provides empirical evidence for your final project defensio!


In [ ]:
# Initialize experiment run with configuration dictionary
run = wandb.init(
    project="cifar10-benchmark-assistant",
    config={
        "architecture": "ModularVGGNet",
        "optimizer": "AdamW",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "epochs": 5,
        "seed": 42
    },
    reinit=True
)

# Log training and validation metrics
wandb.log({
    "epoch": 5,
    "train_loss": avg_loss,
    "val_accuracy": vgg_val_acc,
    "linear_baseline_accuracy": linear_acc,
    "dummy_baseline_accuracy": dummy_acc
})
print("W&B Run Config Architecture:", wandb.config.architecture)
print(f"Logged Epoch 5: Val Acc = {vgg_val_acc*100:.1f}%, Train Loss = {avg_loss:.4f}")


In [ ]:
# Test Split Evaluation and Error Analysis
vgg_model.eval()
with torch.no_grad():
    test_logits = vgg_model(X_test)
    test_preds = test_logits.argmax(dim=1).numpy()
    y_true = test_labels.numpy()

class_names = ["plane", "auto", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

# 1. Concrete numerical classification report
print(classification_report(y_true, test_preds, target_names=class_names, zero_division=0))

# 2. Log confusion matrix artifact to W&B
wandb.log({
    "test_confusion_matrix": wandb.plot.confusion_matrix(
        probs=None, y_true=y_true, preds=test_preds, class_names=class_names
    )
})
wandb.finish()
print("Experiment run successfully completed and logged.")


## 9. Oral Defensio Readiness: The 5-Point Self-Check

- Before submitting your project report and entering the 60% oral defensio, verify that you can answer these five questions without consulting an AI assistant:

1. **Mathematical Objective**: *What loss function is being minimized, and how do parameter updates take place?*
2. **Feature Preprocessing**: *Why was this specific scaler or normalizer chosen, and where in the code is data leakage prevented?*
3. **Hyperparameter Rationale**: *Why were these specific values (e.g. learning rate, weight decay, dropout rate) selected? What was the validation curve?*
4. **Metric Justification**: *Why is your chosen evaluation metric suitable for your domain task and class distribution?*
5. **Failure Analysis**: *When and why does your model make errors? Which instances in the confusion matrix represent costly mistakes?*


### Student Practical Checklist

- [ ] 1. **CONSTITUTION**: Have I defined strict data invariants (channel normalization strictly on train, no pre-split augmentation)?
- [ ] 2. **SPECIFICATION**: Is my learning task formalized as $(T, P, E)$ with explicit baseline benchmarks and targets?
- [ ] 3. **CLARIFICATION**: Have I analyzed edge cases, tensor dimensions, and numerical loss stability?
- [ ] 4. **ARCHITECTURE**: Is my pipeline modular, with decoupled data loading, models, training loops, and W&B tracking?
- [ ] 5. **TASKS**: Are tasks split into blocking data-integrity prerequisites followed by modeling phases?
- [ ] 6. **PRE-EXECUTION AUDIT**: Have I statically checked proposed agent code for silent leakage or missing evaluation modes?
- [ ] 7. **IMPLEMENT (TDD)**: Did the agent write verifiable test assertions (shapes, overfit-on-batch) before training?
- [ ] 8. **EXPERIMENT TRACKING**: Are all parameters, losses, metrics, and confusion matrices logged to W&B?
- [ ] 9. **REVISIONS**: Did I formally revise the plan when training curves revealed severe overfitting or underfitting?
- [ ] 10. **REPRODUCIBILITY**: Are all random seeds pinned, environment dependencies frozen, and Git commits clean?

> 💡 **Key Takeaway:** AI coding agents are productivity multipliers for engineers who understand machine learning principles. Treat the agent as an eager junior assistant: verify every derivation, test every pipeline, and take personal ownership of the code!
